# Titanium alloy aging: reproduce the ML analysis
Upload the `Titanium-Alloy-Aging-ML.zip` package, then run the cells from top to bottom. This recalculates the results using the frozen Excel workbook.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Upload Titanium-Alloy-Aging-ML.zip'
PACKAGE_ZIP = zip_names[0]
print('Uploaded:', PACKAGE_ZIP)


In [ ]:
import zipfile, shutil
from pathlib import Path
WORK = Path('/content/titanium_work')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
with zipfile.ZipFile(PACKAGE_ZIP) as z: z.extractall(WORK)
datasets = list(WORK.rglob('Titanium_Aging_RT_Resistivity_Dataset_Frozen_v1.1_2026-09-20.xlsx'))
assert len(datasets) == 1, f'Expected one dataset, found {len(datasets)}'
DATA = datasets[0]
ROOT = DATA.parent.parent
CODE = ROOT / 'code'
print('Package root:', ROOT)
print('Dataset:', DATA)


In [ ]:
%pip install -q -r "{CODE / 'requirements.txt'}"

## 1. Main resistivity and hardness model comparison

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(CODE/'main_ml'/'train.py'), '--input', str(DATA)], check=True)

## 2. XGBoost and SHAP

In [ ]:
subprocess.run([sys.executable, str(CODE/'main_ml'/'complete_xgb_shap.py')], check=True)

## 3. Paired hardness analysis with measured resistivity

In [ ]:
subprocess.run([sys.executable, str(CODE/'paired_ml'/'run.py'), str(DATA)], check=True)

## 4. Compress and download the newly generated files

In [ ]:
archive = shutil.make_archive('/content/Titanium_ML_Colab_Outputs', 'zip', ROOT)
print('Created:', archive)
files.download(archive)
